In [0]:
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql import functions as F

source_tables = [
    "insurance.homecredit.application_test",
    "insurance.homecredit.application_train",
    "insurance.homecredit.bureau",
    "insurance.homecredit.bureau_balance",
    "insurance.homecredit.previous_application",
    
]

dfs = {table.split('.')[-1]: spark.table(table) for table in source_tables}
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
allowed_bureau_balance_status = ['0', '1', '2', '3', '4', '5', 'C', 'X']
application_universe_df = (
    dfs['application_train'].select('SK_ID_CURR')
    .unionByName(dfs['application_test'].select('SK_ID_CURR'))
    .dropDuplicates()
)

print(f"Validation run id: {run_id}")
display(spark.createDataFrame([(t,) for t in source_tables], ["table_name"]))

Validation run id: 20260627T173844Z


table_name
insurance.homecredit.application_test
insurance.homecredit.application_train
insurance.homecredit.bureau
insurance.homecredit.bureau_balance
insurance.homecredit.previous_application


In [0]:
validation_rows = []

def add_validation_result(validation_id, table_name, validation_name, category, passed, failed_records, checked_records, details):
    validation_rows.append(
        Row(
            run_id=run_id,
            validation_id=int(validation_id),
            table_name=table_name,
            validation_name=validation_name,
            category=category,
            passed=bool(passed),
            failed_records=int(failed_records),
            checked_records=int(checked_records),
            failure_pct=float((failed_records / checked_records) if checked_records else 0.0),
            details=str(details),
            validated_at_utc=datetime.now(timezone.utc).isoformat()
        )
    )
def run_row_count_check(validation_id, table_name):
    checked_records = dfs[table_name].count()
    add_validation_result(
        validation_id,
        table_name,
        f"{table_name} row count greater than zero",
        "volume",
        checked_records > 0,
        0 if checked_records > 0 else 1,
        max(checked_records, 1),
        {"row_count": checked_records}
    )

def run_not_null_check(validation_id, table_name, column_name):
    checked_records = dfs[table_name].count()
    failed_records = dfs[table_name].filter(F.col(column_name).isNull()).count()
    add_validation_result(
        validation_id,
        table_name,
        f"{column_name} contains no nulls",
        "completeness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name}
    )

def run_unique_check(validation_id, table_name, column_name):
    checked_records = dfs[table_name].count()
    distinct_records = dfs[table_name].select(column_name).distinct().count()
    failed_records = checked_records - distinct_records
    add_validation_result(
        validation_id,
        table_name,
        f"{column_name} is unique",
        "uniqueness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name, "distinct_records": distinct_records}
    )


def run_allowed_values_check(validation_id, table_name, column_name, allowed_values):
    checked_df = dfs[table_name].filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    failed_records = checked_df.filter(~F.col(column_name).isin(allowed_values)).count()
    invalid_values = [row[column_name] for row in checked_df.filter(~F.col(column_name).isin(allowed_values)).select(column_name).distinct().limit(10).collect()]
    add_validation_result(
        validation_id,
        table_name,
        f"{column_name} is within the allowed domain",
        "domain",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "allowed_values": allowed_values, "invalid_values": invalid_values}
    )


def run_range_check(validation_id, table_name, column_name, validation_name, condition):
    checked_df = dfs[table_name].filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    failed_records = checked_df.filter(~condition(F.col(column_name))).count()
    stats = checked_df.agg(
        F.min(F.col(column_name)).alias('min_value'),
        F.max(F.col(column_name)).alias('max_value')
    ).collect()[0]
    add_validation_result(
        validation_id,
        table_name,
        validation_name,
        "range",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "min_value": stats['min_value'], "max_value": stats['max_value']}
    )


def run_foreign_key_check(validation_id, child_table_name, child_column, parent_df, parent_column, validation_name):
    child_df = dfs[child_table_name].select(child_column).filter(F.col(child_column).isNotNull())
    checked_records = child_df.count()
    failed_records = (
        child_df.alias('child')
        .join(parent_df.select(parent_column).dropDuplicates().alias('parent'), F.col(f'child.{child_column}') == F.col(f'parent.{parent_column}'), 'left_anti')
        .count()
    )
    add_validation_result(
        validation_id,
        child_table_name,
        validation_name,
        "referential_integrity",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"child_column": child_column, "parent_column": parent_column}
    )


In [0]:
run_row_count_check(1, 'application_train')
run_row_count_check(2, 'application_test')
run_row_count_check(3, 'bureau')
run_row_count_check(4, 'bureau_balance')
run_row_count_check(5, 'previous_application')

run_not_null_check(6, 'application_train', 'SK_ID_CURR')
run_not_null_check(7, 'application_test', 'SK_ID_CURR')
run_not_null_check(8, 'bureau', 'SK_ID_BUREAU')
run_not_null_check(9, 'previous_application', 'SK_ID_PREV')

run_unique_check(10, 'application_train', 'SK_ID_CURR')
run_unique_check(11, 'application_test', 'SK_ID_CURR')
run_unique_check(12, 'bureau', 'SK_ID_BUREAU')

run_allowed_values_check(13, 'application_train', 'TARGET', [0.0, 1.0])

run_range_check(14, 'application_train', 'AMT_CREDIT', 'application_train AMT_CREDIT is positive when present', lambda c: c > F.lit(0))
run_range_check(15, 'application_train', 'AMT_INCOME_TOTAL', 'application_train AMT_INCOME_TOTAL is positive when present', lambda c: c > F.lit(0))
run_allowed_values_check(16, 'bureau_balance', 'STATUS', allowed_bureau_balance_status)
run_foreign_key_check(17, 'previous_application', 'SK_ID_CURR', application_universe_df, 'SK_ID_CURR', 'previous_application SK_ID_CURR exists in the application universe')

all_validation_results_df = spark.createDataFrame(validation_rows).orderBy('validation_id')
failed_validation_results_df = all_validation_results_df.filter(~F.col('passed')).orderBy('validation_id')
     

In [0]:
validation_summary_df = all_validation_results_df.groupBy('run_id').agg(
    F.count('*').alias('total_validations'),
    F.sum(F.when(F.col('passed'), 1).otherwise(0)).alias('passed_validations'),
    F.sum(F.when(~F.col('passed'), 1).otherwise(0)).alias('failed_validations')
)

validation_results = {
    table_name: all_validation_results_df.filter(F.col('table_name') == table_name).orderBy('validation_id')
    for table_name in sorted([row['table_name'] for row in all_validation_results_df.select('table_name').distinct().collect()])
}

print('Detailed validation results')
display(all_validation_results_df)

print('Validation summary')
display(validation_summary_df)

print('Failed validations')
display(failed_validation_results_df)
     

Detailed validation results


run_id,validation_id,table_name,validation_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc
20260627T173844Z,1,application_train,application_train row count greater than zero,volume,true,0,307511,0.0,{'row_count': 307511},2026-06-27T17:38:59.493463+00:00
20260627T173844Z,2,application_test,application_test row count greater than zero,volume,true,0,48744,0.0,{'row_count': 48744},2026-06-27T17:38:59.737851+00:00
20260627T173844Z,3,bureau,bureau row count greater than zero,volume,true,0,1716428,0.0,{'row_count': 1716428},2026-06-27T17:38:59.970301+00:00
20260627T173844Z,4,bureau_balance,bureau_balance row count greater than zero,volume,true,0,27299925,0.0,{'row_count': 27299925},2026-06-27T17:39:00.204736+00:00
20260627T173844Z,5,previous_application,previous_application row count greater than zero,volume,true,0,1670214,0.0,{'row_count': 1670214},2026-06-27T17:39:00.462744+00:00
20260627T173844Z,6,application_train,SK_ID_CURR contains no nulls,completeness,true,0,307511,0.0,{'column_name': 'SK_ID_CURR'},2026-06-27T17:39:00.964127+00:00
20260627T173844Z,7,application_test,SK_ID_CURR contains no nulls,completeness,true,0,48744,0.0,{'column_name': 'SK_ID_CURR'},2026-06-27T17:39:01.475880+00:00
20260627T173844Z,8,bureau,SK_ID_BUREAU contains no nulls,completeness,true,0,1716428,0.0,{'column_name': 'SK_ID_BUREAU'},2026-06-27T17:39:01.957495+00:00
20260627T173844Z,9,previous_application,SK_ID_PREV contains no nulls,completeness,true,0,1670214,0.0,{'column_name': 'SK_ID_PREV'},2026-06-27T17:39:02.463699+00:00
20260627T173844Z,10,application_train,SK_ID_CURR is unique,uniqueness,true,0,307511,0.0,"{'column_name': 'SK_ID_CURR', 'distinct_records': 307511}",2026-06-27T17:39:03.164681+00:00


Validation summary


run_id,total_validations,passed_validations,failed_validations
20260627T173844Z,17,17,0


Failed validations


run_id,validation_id,table_name,validation_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc


In [0]:
results_table_name = 'insurance.homecredit.validation_results_silver'
summary_table_name = 'insurance.homecredit.validation_summary_silver'
failed_table_name = 'insurance.homecredit.validation_failed_silver'

all_validation_results_df.write.format('delta').mode('append').saveAsTable(results_table_name)
validation_summary_df.write.format('delta').mode('append').saveAsTable(summary_table_name)
failed_validation_results_df.write.format('delta').mode('append').saveAsTable(failed_table_name)

artifacts_df = spark.createDataFrame([
    ('results_table', results_table_name),
    ('summary_table', summary_table_name),
    ('failed_table', failed_table_name),
    ('run_id', run_id)
], ['artifact_type', 'artifact_location'])

display(artifacts_df)
     


artifact_type,artifact_location
results_table,insurance.homecredit.validation_results_silver
summary_table,insurance.homecredit.validation_summary_silver
failed_table,insurance.homecredit.validation_failed_silver
run_id,20260627T173844Z


In [0]:
from azure.storage.blob import BlobServiceClient
import io
import pandas as pd

connection_string = "DefaultEndpointsProtocol=https;AccountName=dbq2source;AccountKey=PXXhAoeMGDPApjyShkzLheM345zVnXduPfGO3yQT0RyCOqce4YTlt5Xi3ET7mShSXhNdvxi/l5yt+AStfnKUVw==;EndpointSuffix=core.windows.net"
container_name = 'source'
log_folder = 'logs'

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

In [0]:
for name, result_df in validation_results.items():
    pdf = result_df.toPandas()
    csv_payload = pdf.to_csv(index=False)
    log_blob_name = f"{log_folder}{name}.csv"

    try:
        existing_data = container_client.download_blob(log_blob_name).readall().decode('utf-8')
        if existing_data.strip():
            appended_data = existing_data.rstrip('\n') + '\n' + pdf.to_csv(index=False, header=False)
        else:
            appended_data = csv_payload
    except Exception:
        appended_data = csv_payload

    container_client.upload_blob(log_blob_name, appended_data, overwrite=True)
    print(f"Appended log file: {log_blob_name}")

Appended log file: logsapplication_test.csv
Appended log file: logsapplication_train.csv
Appended log file: logsbureau.csv
Appended log file: logsbureau_balance.csv
Appended log file: logsprevious_application.csv
